## Setup: connect to PostgreSQL

Loads database credentials from `.env` and opens a SQLAlchemy engine.
Reused by every query cell below.

In [58]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd

# Credentials come from .env (gitignored) so they never reach the repo
load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

pd.read_sql("SELECT version();", engine)

,version
0,"PostgreSQL 18.4 on x86_64-apple-darwin24.6.0, ..."


In [59]:
# Exploring ASHE structure
import pandas as pd

path = "../data/ashe/ashe_2025/PROV - Home Geography Table 8.7a   Annual pay - Gross 2025.xlsx"

xl = pd.ExcelFile(path)
print(xl.sheet_names)

pd.read_excel(path, sheet_name=1, header=None).head(25) # smoke test: confirms the connection works


['Notes', 'All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Time', 'Female Part-Time']


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,Table 8.7a Annual pay - Gross (£) - For all ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Number,NaN,Annual,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,of jobsb,NaN,percentage,NaN,percentage,Percentiles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Description,Code,(thousand),Median,change,Mean,change,10,20,25,30,40,60,70,75,80,90,NaN,NaN,NaN
5,United Kingdom,K02000001,24897,32890,4.1,40269,5.5,11425,18560,22060,24532,28591,38000,44500,48283,52809,69381,NaN,NaN,NaN
6,Great Britain,K03000001,23879,32978,3.9,40436,5.4,11480,18625,22120,24585,28650,38063,44635,48403,52923,69719,NaN,Key,Statistical robustness
7,England and Wales,K04000001,21674,32963,4,40681,5.4,11429,18579,22080,24547,28617,38034,44654,48447,53123,70138,NaN,CV <= 5%,Estimates are considered precise
8,England,E92000001,20472,33080,3.9,41038,5.3,11440,18624,22142,24632,28722,38223,44898,48731,53546,70943,NaN,CV > 5% and <= 10%,Estimates are considered reasonably precise
9,North East,E12000001,882,29584,3.6,33999,4.4,10982,17650,20833,23233,26427,33755,38994,42371,46133,57592,NaN,CV > 10% and <= 20%,Estimates are considered acceptable


In [60]:
pd.read_excel(path, sheet_name=1, header=None).tail(10)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
396,Stirling,S12000030,40,33832,-0.7,43262,2.5,x,20705,23393,26431,30173,39474,47630,50055,x,x,NaN,NaN,NaN
397,West Dunbartonshire,S12000039,35,31962,11.2,35428,9.4,12655,19576,22931,24869,27548,37897,43900,46154,49555,x,NaN,NaN,NaN
398,West Lothian,S12000040,79,33302,-3.6,38508,2.2,13125,21446,24110,25624,29210,39748,44571,48046,51208,x,NaN,NaN,NaN
399,Northern Ireland,N92000002,883,31252,8.6,35789,9.3,10299,17528,21026,23618,27518,35877,41221,44657,48950,61572,NaN,NaN,NaN
400,Not Classified,NaN,136,31284,0.6,40154,6.3,8640,15512,19057,22458,27204,36579,43578,48694,53567,x,NaN,NaN,NaN
401,a Employees on adult rates who have been in t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
402,b Figures for Number of Jobs are for indicati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
403,KEY - The colour coding indicates the quality ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404,The quality of an estimate is measured by its ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
405,"Source: Annual Survey of Hours and Earnings, O...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- Rows 0 - 4 are junk. 

- Rows 5 onwards starts at United Kingdom, Great Britain, Emnland and Wales, England, then regions and local authorities.

Now must find which rows are regions. By eye there are 9 English regions plus Wales, Scotland, and Northern Ireland. We must disregard the local authorities.

In [61]:
df = pd.read_excel(path, sheet_name= 'All', header = None)
mask = df[0].isin(['Wales','Scotland','Northern Ireland','London'])
df[mask][[0,1,3]]

,0,1,3


In [62]:
london_rows = df[df[0].astype(str).str.contains('London', na=False)]
print([repr(x) for x in london_rows[0].tolist()])

["'London '", "'Inner London'", "'  City of London'", "'Outer London'"]


In [63]:
df[0] = df[0].astype(str).str.strip()
df[1] = df[1].astype(str).str.strip()
df[df[0].isin(['Wales', 'Scotland', 'Northern Ireland', 'London'])][[0, 1, 3]]

,0,1,3
206,London,E12000007,39778
343,Wales,W92000004,30732
366,Scotland,S92000003,33061
399,Northern Ireland,N92000002,31252


In [64]:
regions = df[df[1].str.startswith(('E12', 'W92', 'S92', 'N92'), na=False)][[0, 1, 3]]
regions

,0,1,3
9,North East,E12000001,29584
23,North West,E12000002,31330
62,Yorkshire and The Humber,E12000003,30682
80,East Midlands,E12000004,30690
120,West Midlands,E12000005,31345
155,East,E12000006,34104
206,London,E12000007,39778
242,South East,E12000008,35215
313,South West,E12000009,31432
343,Wales,W92000004,30732


Does an old .xls file have the same layout as a .xlsx file?

In [65]:
old_path = "../data/ashe/ashe_2005/Home Geography Table 8.7a   Annual pay - Gross 2005.xls"
xl_old = pd.ExcelFile(old_path)
print(xl_old.sheet_names)

['All', 'Male', 'Female', 'Full-Time', 'Part-Time', 'Male Full-Time', 'Male Part-Time', 'Female Full-Time', 'Female Part-Time']


In [66]:
df_old = pd.read_excel(old_path, sheet_name=0, header=None)
df_old.head(12)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,Table 8.7a Annual pay - Gross (£) - For all ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Number,NaN,Annual,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,of jobsb,NaN,percentage,NaN,percentage,Percentiles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Description,Code,(thousand),Median,change,Mean,change,10,20,25,30,40,60,70,75,80,90,NaN,NaN
5,United Kingdom,NaN,19114,18949,4.3,23389,5.1,5874,9795,11514,13079,15964,22383,26432,28869,31579,40564,NaN,Key
6,Great Britain,NaN,18194,19093,4.5,23592,5.2,5927,9907,11647,13217,16098,22558,26605,29040,31782,40908,NaN,CV <= 5%
7,England and Wales,NaN,16454,19235,4.3,23853,5.1,5920,9945,11699,13275,16204,22709,26780,29250,32015,41335,NaN,CV > 5% and <= 10%
8,England,NaN,15576,19364,4.3,24062,5,5933,9981,11744,13345,16299,22840,26948,29422,32208,41733,NaN,CV > 10% and <= 20%
9,North East,NaN,773,16942,5.8,19765,6.4,5889,9408,10983,12245,14444,19632,23488,25407,28088,34636,NaN,x = unreliable


In [67]:
df_old[df_old[0].str.contains('Wales', na=False)][[0, 3]]

,0,3
7,England and Wales,19235
414,Wales / Cymru,17253


In [68]:
regions_xls = [
    'North East', 'North West', 'Yorkshire and The Humber',
    'East Midlands', 'West Midlands', 'East', 'London',
    'South East', 'South West', 'Wales / Cymru', 'Scotland', 'Northern Ireland'
]

df_old[0] = df_old[0].astype(str).str.strip()
df_old[df_old[0].isin(regions_xls)][[0, 3]]

,0,3
9,North East,16942
36,North West,18140
85,Yorkshire and The Humber,17731
110,East Midlands,18201
156,West Midlands,18020
196,East,19993
251,London,24704
287,South East,20902
362,South West,17808
414,Wales / Cymru,17253


Testing if all 24 years can parse into 12 regions, now that we know both 2025 and 2005 files work. 

In [69]:
import glob

REGIONS = [
    'North East', 'North West', 'Yorkshire and The Humber',
    'East Midlands', 'West Midlands', 'East', 'London',
    'South East', 'South West', 'Wales', 'Scotland', 'Northern Ireland'
]

RENAME = {
    'Wales / Cymru': 'Wales',
}

for path in sorted(glob.glob("../data/ashe/*/*8.7a*")):
    year = path.split('/')[-2]
    d = pd.read_excel(path, sheet_name='All', header=None)
    d[0] = d[0].astype(str).str.strip().replace(RENAME)
    found = set(d[d[0].isin(REGIONS)][0])
    missing = set(REGIONS) - found
    print(f"{year}: {len(found)} regions", f"MISSING: {missing}" if missing else "")

ashe_2002: 12 regions 
ashe_2003: 12 regions 
ashe_2004: 12 regions 
ashe_2005: 12 regions 
ashe_2006: 12 regions 
ashe_2007: 12 regions 
ashe_2008: 12 regions 
ashe_2009: 12 regions 
ashe_2010: 12 regions 
ashe_2011: 12 regions 
ashe_2012: 12 regions 
ashe_2013: 12 regions 
ashe_2014: 12 regions 
ashe_2015: 12 regions 
ashe_2016: 12 regions 
ashe_2017: 12 regions 
ashe_2018: 12 regions 
ashe_2019: 12 regions 
ashe_2020: 12 regions 
ashe_2021: 12 regions 
ashe_2022: 12 regions 
ashe_2023: 12 regions 
ashe_2024: 12 regions 
ashe_2025: 12 regions 


Building the earnings pandas DataFrame

In [70]:
import glob
import re

REGIONS = [
    'North East', 'North West', 'Yorkshire and The Humber',
    'East Midlands', 'West Midlands', 'East', 'London',
    'South East', 'South West', 'Wales', 'Scotland', 'Northern Ireland'
]

RENAME = {'Wales / Cymru': 'Wales'}


def parse_ashe(path):
    """Read one ASHE Table 8.7a file, return 12 regions with median pay and year."""
    year = int(re.search(r'ashe_(\d{4})', path).group(1))

    df = pd.read_excel(path, sheet_name='All', header=None)
    df[0] = df[0].astype(str).str.strip().replace(RENAME)

    out = df[df[0].isin(REGIONS)][[0, 3]].copy()
    out.columns = ['region', 'median_pay']
    out['year'] = year

    return out


files = sorted(glob.glob("../data/ashe/*/*8.7a*"))
earnings = pd.concat([parse_ashe(f) for f in files], ignore_index=True)

print(earnings.shape)
earnings

(288, 3)


,region,median_pay,year
0,North East,14798,2002
1,North West,16081,2002
2,Yorkshire and The Humber,15711,2002
3,East Midlands,16218,2002
4,West Midlands,16243,2002
...,...,...,...
283,South East,35215,2025
284,South West,31432,2025
285,Wales,30732,2025
286,Scotland,33061,2025


In [71]:
earnings.dtypes

region           str
median_pay    object
year           int64
dtype: object

In [72]:
earnings['median_pay'].describe()

count       288
unique      283
top       20000
freq          3
Name: median_pay, dtype: int64

In [73]:
earnings[pd.to_numeric(earnings['median_pay'], errors='coerce').isna()]

,region,median_pay,year


In [74]:
earnings['median_pay'] = earnings['median_pay'].astype(int)
earnings.dtypes

region          str
median_pay    int64
year          int64
dtype: object

In [75]:
earnings['median_pay'].describe()

count      288.000000
mean     22667.656250
std       4733.261984
min      14798.000000
25%      19360.750000
50%      21778.000000
75%      25360.500000
max      39778.000000
Name: median_pay, dtype: float64

In [76]:
earnings[earnings['region'] == 'London'].sort_values('year')

,region,median_pay,year
7,London,22000,2002
19,London,22983,2003
30,London,23595,2004
42,London,24704,2005
54,London,25000,2006
66,London,26278,2007
78,London,27327,2008
90,London,28051,2009
102,London,27721,2010
114,London,27420,2011


In [77]:
earnings.to_sql('earnings', engine, if_exists='replace', index = False)

288

In [81]:
pd.read_sql('SELECT * FROM earnings ORDER BY YEAR, region LIMIT 12;', engine)

,region,median_pay,year
0,East,17816,2002
1,East Midlands,16218,2002
2,London,22000,2002
3,North East,14798,2002
4,North West,16081,2002
5,Northern Ireland,15314,2002
6,Scotland,15820,2002
7,South East,19096,2002
8,South West,15631,2002
9,Wales,15000,2002
